# NB-05: Figure & Image Dependency Checker

Scans all `\\includegraphics` calls, detects `\\fbox` placeholders, checks whether required image files exist on disk, and audits every figure environment for labels and captions.

## Step 1 — Extract all \\includegraphics paths

In [ ]:
import re, pathlib, collections
from pathlib import Path

TEX_FILE = "jmlr-hypatiax-paper-final.tex"
source = Path(TEX_FILE).read_text(encoding="utf-8")
lines  = source.splitlines()

# ---- Extract \includegraphics paths ----
INCL_RE = re.compile(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}')
fig_refs = INCL_RE.findall(source)
print(f"Total \\includegraphics calls: {len(fig_refs)}")
for f in fig_refs:
    print(f"  {f}")

## Step 2 — Detect \\fbox placeholder figures

In [ ]:
# ---- Check for fbox placeholder figures ----
fbox_lines = [(i+1, ln.strip()) for i, ln in enumerate(lines) if r'\fbox' in ln]
print(f"\\fbox placeholder occurrences: {len(fbox_lines)}")
for lno, ctx in fbox_lines:
    print(f"  line {lno}: {ctx[:100]}")

## Step 3 — Check required image files on disk

In [ ]:
# ---- Check which image files exist on disk ----
# Known image directories from \graphicspath in preamble
SEARCH_DIRS = [
    Path("figures"),
    Path("../figures"),
    Path("."),
]

# Known required images from figures/tables inventory
REQUIRED_IMAGES = {
    "hypatiaX_three_systems":         "fig:architecture  (§7.1)  — PLACEHOLDER",
    "fig18_r2_heatmap_improved":      "fig:r2_heatmap_clipped  (§10.2)",
    "fig09_r2_heatmap_regimes":       "fig:r2_heatmap_raw  (§10.2)",
    "fig1_seed_sweep":                "fig:portfolio_seed_sweep  (§10.5)",
    "hypatiaX_algorithm1_routing_cascade_v2": "fig:routing_cascade  (§7.4)  — algorithm flow",
}

EXTENSIONS = [".pdf", ".png", ".jpg", ".eps", ".svg"]

print("Image file availability check:")
print("-" * 80)
for stem, desc in REQUIRED_IMAGES.items():
    found = False
    found_path = None
    for d in SEARCH_DIRS:
        for ext in EXTENSIONS:
            candidate = d / (stem + ext)
            if candidate.exists():
                found = True
                found_path = candidate
                break
        if found:
            break
    status = f"FOUND at {found_path}" if found else "MISSING"
    flag   = "OK" if found else "WARNING"
    print(f"  [{flag}]  {stem}")
    print(f"          {desc}")
    print(f"          Status: {status}")
    print()

## Step 4 — Full figure environment audit

In [ ]:
# ---- Audit all figure environments for labels and captions ----
fig_blocks = re.findall(
    r'\\begin\{figure\*?\}(.*?)\\end\{figure\*?\}',
    source, re.DOTALL)
print(f"Figure environments found: {len(fig_blocks)}")
print("-" * 80)
for i, block in enumerate(fig_blocks):
    label_m = re.search(r'\\label\{([^}]+)\}', block)
    cap_m   = re.search(r'\\caption\{([^}]{0,80})', block)
    incl_m  = re.search(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}', block)
    fbox_m  = r'\fbox' in block

    label = label_m.group(1) if label_m else "[NO LABEL]"
    cap   = cap_m.group(1).strip()[:60] if cap_m else "[NO CAPTION]"
    img   = incl_m.group(1) if incl_m else ("[fbox PLACEHOLDER]" if fbox_m else "[NO IMAGE]")

    flag = "WARNING - PLACEHOLDER" if fbox_m or (not incl_m) else "OK"
    print(f"  Fig {i+1:2d}  [{flag}]")
    print(f"          label  : {label}")
    print(f"          image  : {img}")
    print(f"          caption: {cap}...")
    print()

## Step 5 — Fix recipe

In [ ]:
fixes = (
    "FIX-F1  hypatiaX_three_systems — Architecture figure (fig:architecture)\n"
    "  Currently: \\fbox{...} placeholder in Section 7.1.\n"
    "  ACTION: Replace placeholder with final PDF/PNG from\n"
    "  Figures/architecture_figures/ in the jmlr tree.\n"
    "  File should be placed in figures/ directory.\n\n"
    "FIX-F2  fig18_r2_heatmap_improved — R^2 heatmap clipped (fig:r2_heatmap_clipped)\n"
    "  Source: Figures/figures-cosmetic-last/fig18_r2_heatmap_improved.pdf\n"
    "  ACTION: Copy to figures/ directory.\n\n"
    "FIX-F3  fig09_r2_heatmap_regimes — R^2 heatmap raw (fig:r2_heatmap_raw)\n"
    "  Source: Figures/figures-cosmetic-last/fig09_r2_heatmap_regimes.pdf\n"
    "  ACTION: Copy to figures/ directory.\n\n"
    "FIX-F4  fig1_seed_sweep — Portfolio Variance seed sweep (fig:portfolio_seed_sweep)\n"
    "  Source: Figures/figures-portfolio-variance/fig1_seed_sweep.pdf (or .png)\n"
    "  ACTION: Copy to figures/ directory.\n\n"
    "NOTE: fig:routing_cascade (hypatiaX_algorithm1_routing_cascade_v2)\n"
    "  is marked as present in the inventory (status: OK B-07 addressed).\n"
    "  Verify the file exists and compiles correctly.\n"
)
print(fixes)